# R45 reconstructed supplementary-frame pilot

Use **L4 or A100 GPU** in Colab. Run the source/preflight cells, then the four-batch trial. If they pass, run both 30-epoch arms and return the final ZIP.

This replacement is not the lost original source. It uses the exact published R37 effective labels and R44 complete frame_0383. Control: 137 photos / 3,784 fronts. Candidate: 138 photos / 3,827 fronts. Both independently start from original epoch173. All 20 original trainer-validation photos remain unchanged; three reference sets are scored separately. The other eleven R44 photos stay held. No GPU result or promotion is claimed. One additional view is a development pilot, not an independent test.

The native seed46 recipe, 1280 input, batch4, AdamW, FP32, 30 epochs and final last.pt selection are unchanged. The measured budgets must be 1,050 batches / 78 optimizer updates per arm and 4,110 versus 4,140 image presentations.


In [ ]:
import os, json, hashlib, subprocess, shutil, sys
from pathlib import Path
from importlib.metadata import version

SOURCE_REVISION = "ea4ac955f2dfaf1add24c243599715f802eb41a7"
BASE = Path(os.environ.get("BAOMAI_R45_REBUILT_BASE", "/content/baomai-r45-rebuilt")).resolve()
SOURCE = BASE / "source"
WORK = BASE / "work"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE":"1", "GIT_TERMINAL_PROMPT":"0"}
def run(args, cwd=None, env=None):
    print("+", " ".join(map(str,args)), flush=True)
    subprocess.run(list(map(str,args)), cwd=cwd, env=env, check=True)
def sha(path):
    with Path(path).open("rb") as f:
        return hashlib.file_digest(f,"sha256").hexdigest()

if subprocess.run(["git","lfs","version"],capture_output=True).returncode:
    run(["apt-get","update","-qq"])
    run(["apt-get","install","-y","-qq","git-lfs"])
if not SOURCE.exists():
    run(["git","init",SOURCE],env=ENV)
    run(["git","remote","add","origin","https://github.com/Siuuuers/dl.git"],cwd=SOURCE)
    run(["git","lfs","install","--local"],cwd=SOURCE)
    run(["git","config","remote.origin.promisor","true"],cwd=SOURCE)
    run(["git","config","remote.origin.partialclonefilter","blob:none"],cwd=SOURCE)
    run(["git","fetch","--depth=1","--filter=blob:none","origin",SOURCE_REVISION],cwd=SOURCE,env=ENV)
    run(["git","sparse-checkout","init","--no-cone"],cwd=SOURCE)
    run(["git","sparse-checkout","set","--no-cone","/.gitattributes","/scripts/","/data/manifest.json",
         "/data/images/train/","/data/images/val/","/data/labels/","/weights/cassette_lower_epoch173.pt",
         "/data/candidates/front-review-2026-11-26-r37/","/data/reviews/fronts-2026-11-26-r38/",
         "/data/references/sequence-refined-reference-2026-11-18/","/data/references/visible-front-completion-2026-11-14/",
         "/results/review/front-mining-r44/review.zip"],cwd=SOURCE)
    run(["git","checkout","--detach",SOURCE_REVISION],cwd=SOURCE,env=ENV)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=SOURCE,text=True).strip()==SOURCE_REVISION, "Existing checkout differs; preserve it and choose a fresh base."
run(["git","lfs","pull","--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt","--exclude="],cwd=SOURCE)
assert not subprocess.check_output(["git","status","--porcelain","--untracked-files=no"],cwd=SOURCE,text=True).strip(), "Tracked source changed."
SCRIPT=SOURCE/"scripts/rebuild_additional_front_pilot_r45.py"
compile(SCRIPT.read_text(),str(SCRIPT),"exec")
print("Pinned source:", SOURCE_REVISION)


## Verify input bytes, full annotation inventory, and reference membership

This can be run without training. Stop on any error. The first real GPU trial is in the next section.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select an L4 or A100 GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
constraints=BASE/"torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch","torchvision")))
run([sys.executable,"-m","pip","install","-c",constraints,"ultralytics==8.4.182","psutil"])
def command(action, *extra):
    return [sys.executable,"-B",SCRIPT,action,"--root",SOURCE,"--work",WORK,*extra]
run(command("prepare"))
print("Data preflight complete. No training has occurred.")


## Backup completed arms (optional Drive)

This saves only the final checkpoints, training receipts, progress, CSVs and logs from each complete arm. A completed incompatible arm is preserved, not silently overwritten. For an interrupted run, rename the incomplete folder and retry; never delete a completed checkpoint. Use the same GPU/runtime package versions for both arms.


In [ ]:
USE_DRIVE = True
BACKUP = None
if USE_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        print("Drive unavailable")
    else:
        drive.mount("/content/drive")
        BACKUP=Path("/content/drive/MyDrive/baomai-r45-rebuilt")/SOURCE_REVISION
        BACKUP.mkdir(parents=True,exist_ok=True)
IDENTITY = sha(WORK/"staging.json")

def selected(name):
    return ["receipt.json","progress.json"] + ([] if name=="resource-trial" else ["fit/results.csv","fit/weights/last.pt"])

def validate_backup(folder,name):
    mark=json.loads((folder/"backup.json").read_text())
    assert mark["staging_sha256"]==IDENTITY and mark["source"]==SOURCE_REVISION
    assert set(mark["files"])==set(selected(name))
    for rel,h in mark["files"].items(): assert sha(folder/rel)==h, rel
    if mark.get("log_sha256"): assert sha(folder/"run.log")==mark["log_sha256"]
    return mark

def restore(name):
    if not BACKUP or (WORK/name).exists() or not (BACKUP/name/"backup.json").exists(): return
    source=BACKUP/name
    mark=validate_backup(source,name)
    for rel in selected(name):
        dest=WORK/name/rel
        dest.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(source/rel,dest)
    if mark.get("log_sha256"): shutil.copyfile(source/"run.log",WORK/(name+".log"))
    print("Restored completed arm",name)

def backup(name):
    if not BACKUP: return
    source=WORK/name
    dest=BACKUP/name
    marker={"source":SOURCE_REVISION,"staging_sha256":IDENTITY,
            "files":{r:sha(source/r) for r in selected(name)}}
    log=WORK/(name+".log")
    if log.is_file(): marker["log_sha256"]=sha(log)
    if (dest/"backup.json").exists():
        assert validate_backup(dest,name)==marker, "Different completed backup preserved"
        return
    assert not dest.exists(), "Incomplete Drive backup preserved; rename it before retry"
    dest.mkdir()
    for rel in selected(name):
        p=dest/rel; p.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(source/rel,p)
    if log.is_file(): shutil.copyfile(log,dest/"run.log")
    (dest/"backup.json").write_text(json.dumps(marker,indent=2)+"\n")
    validate_backup(dest,name)
    print("Saved completed arm",name)

def execute(name,action,role):
    restore(name)
    if (WORK/name/"receipt.json").exists():
        run(command(action,"--role",role))
    else:
        log=WORK/(name+".log")
        with log.open("a",encoding="utf-8") as f:
            with subprocess.Popen(list(map(str,command(action,"--role",role))),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
                for line in p.stdout:
                    print(line,end="",flush=True); f.write(line); f.flush()
                if p.wait(): raise RuntimeError(f"{name} failed; see {log}")
    backup(name)


## Discarded resource trial

This is the first actual model/loader check on the reconstruction. It must finish four batches with finite gradients and a real parameter update. Its weights are not used. Do not continue if it fails.


In [ ]:
execute("resource-trial","trial","candidate")


## Run both fresh 30-epoch training arms

Each starts independently from epoch173. The candidate run checks the reference runtime and receipts. Backups happen between arms.


In [ ]:
execute("reference","train","reference")
execute("candidate","train","candidate")


## Fixed evaluation and result archive

Read raw and postclip misses, duplicate outputs, other unmatched outputs, and corner successes together. Missed references remain in corner-success denominators. No change of model selection follows automatically. Return the ZIP, not only a screenshot of the counts.


In [ ]:
run(command("evaluate"))
ARCHIVE=BASE/"baomai-additional-front-pilot-r45-rebuilt-results.zip"
if not ARCHIVE.exists():
    run(command("package","--output",ARCHIVE))
else:
    print("Existing return archive preserved; verify it before returning.")
summary=json.loads((WORK/"evaluation/summary.json").read_text())
for ref,report in summary["references"].items():
    for view in ("raw","postclip"):
        for role in ("reference","candidate"):
            d=report[view][role]["detection"]
            c=report[view][role]["corners_by_locatable_count"]["2"]
            print(ref,view,role,"misses",d["fn"],"unmatched",d["fp"],"duplicates",d["duplicate_fp"],"other",d["other_fp"],"two corners within5",c["within5"],"/",c["eligible"],"within10",c["within10"])
if BACKUP:
    dest=BACKUP/ARCHIVE.name
    if dest.exists(): assert sha(dest)==sha(ARCHIVE), "Different return preserved"
    else: shutil.copyfile(ARCHIVE,dest)
    assert sha(dest)==sha(ARCHIVE)
print("Return:",ARCHIVE,"SHA256:",sha(ARCHIVE))
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(ARCHIVE))
